In [17]:
import pandas as pd
import numpy as np

# 🛒 Olist Data Architecture EDA
### **The Logic of a Marketplace Database**

---

## **1. The Identity Layer (Who & When)**
* **`customer_unique_id`**: **The Actual Human**
  * **Use:** Track long-term loyalty and repeat purchases. $1 \text{ Human} = 1 \text{ Unique ID}$.
* **`customer_id`**: **The Session Token**
  * **Use:** A temporary ID created at every checkout. If a person buys from 3 sellers in one "trip," they get 3 different `customer_id`s in this dataset which also means 3 different `order_id`s).
* **`order_purchase_timestamp`**: **The Moment of Intent**
  * **Note:** If multiple `order_id`s have the same timestamp (or $\Delta t \le 1s$), they were part of the **same shopping cart**.

---

## **2. The Logistics Layer (The "Box")**
* **`order_id`**: **The Fulfillment Contract.**
  * **1 order_id = 1 customer_id = 1 seller = 1 box with (1, ..., k) items**
  * Represents a **single seller's responsibility**. If you buy from two sellers, you get two `order_id`s.
  * **Unique in `order_items`**: The seller is shipping $n=1$ physical unit.
  * **Duplicated in `order_items`**: The seller is shipping $n>1$ units in one box.
* **`seller_id`**: **The Business Partner.** * The independent store that owns the stock and ships the item.
* **`shipping_limit_date`**: The "deadline" for the seller to hand the box to the post office.

---

## **3. The Product Layer (The "Stuff")**
* **`product_id`**: **The Catalog Item.** (e.g., "Blue iPhone 13").
* **`order_item_id`**: **The Box Sequence.**
  * 1 unit = 1 row (even if customer bought x of the same product)
  * If a box has $k$ items, they are indexed $i \in \{1, 2, \dots, k\}$ within that single `order_id` (same seller).
* **`price`**: The cost of $1$ unit ($P$).
* **`freight_value`**: The shipping cost ($F$) for each **`order_item_id`** .

---

## **4. The Feedback & Finance Layer**
* **`review_id`**: **The Customer's Voice.**
  * **Unique**: customer is reviewing a single box (i.e., contains one or more items, but from the same seller)
  * **Duplicated**: customer is reviewing multiple boxes(i.e., bought item(s)from at least two different sellers in one order
* **`payment_sequential`**: Used if a customer split their total across multiple methods.  
  $$\text{Total Order Value} = \sum_{i=1}^{n} \text{payment\_value}_i$$
* **`payment_installments`**: The number of monthly installments ($m$) chosen by the customer.  
  $$\text{Monthly Installment} = \frac{\text{payment\_value}}{m}$$

---

## **🎯 The "Golden Rules" for Data Transformation**

| Scenario | Data Pattern | Reality |
| :--- | :--- | :--- |
| **Simple Buy** | $1 \text{ Order} + 1 \text{ Item Row}$ | 1 Person bought 1 thing. |
| **Big Box** | $1 \text{ Order} + k \text{ Item Rows}$ | 1 Seller shipping $k$ things in **one box**. |
| **Split Cart** | $n \text{ Orders} + 1 \text{ Review ID}$ | 1 Person buying from **$n$ different sellers**. |

---

> **Expert Tip for Gold Layer:** To find the "True" number of shopping trips, calculate $\text{count}(\text{distinct } \text{review\_id})$. Counting `order_id` alone will **inflate** your sales metrics because the marketplace shreds a single cart into multiple seller contracts.

---
**olist_customers_dataset**:



In [18]:
customers = pd.read_csv("olist_customers_dataset.csv")
customers

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP
...,...,...,...,...,...
99436,17ddf5dd5d51696bb3d7c6291687be6f,1a29b476fee25c95fbafc67c5ac95cf8,3937,sao paulo,SP
99437,e7b71a9017aa05c9a7fd292d714858e8,d52a67c98be1cf6a5c84435bd38d095d,6764,taboao da serra,SP
99438,5e28dfe12db7fb50a4b2f691faecea5e,e9f50caf99f032f0bf3c55141f019d99,60115,fortaleza,CE
99439,56b18e2166679b8a959d72dd06da27f9,73c2643a0a458b49f58cea58833b192e,92120,canoas,RS


In [19]:
print("customers_dataset:\n")
print(customers.columns.tolist())
print("\n")

print("Null Count:")
for col in customers.columns.tolist():
  print(f'{col}: {customers[col].isna().sum()}')
print("\n")

print(f'Customer_ID Unique: {customers['customer_id'].nunique()/customers.shape[0] * 100}% ({customers['customer_id'].nunique()} unique)')
print(f'Customer_Unique_ID: {customers['customer_unique_id'].nunique()/customers.shape[0] * 100}% ({customers['customer_unique_id'].nunique()} unique)')
print("""Meaning:
A single 'Buy' click (Cart) is shredded into multiple customer_id's if there are multiple sellers.
This is why there are duplicate customer_unique_id's which more customer_id's
""")
print("Example: 1 Cart + 3 Sellers = 3 customer_id's + 3 order_id's + 1 customer_unique_id.")

print("\n--- PIPELINE STRATEGY ---")
print("Bronze/Silver: Use customer_id/order_id to preserve the Seller-to-Item relationship (1:1:1:1).")
print("Gold (Transactions): Group by [customer_unique_id + timestamp] to see 'True Carts'.")
print("Gold (LTV): Group by [customer_unique_id] ONLY to see 'Lifetime Value' across all time.\n")

print("Customer City & State Counts:")
display(customers['customer_city'].value_counts())
display(customers['customer_state'].value_counts())

customers_dataset:

['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']


Null Count:
customer_id: 0
customer_unique_id: 0
customer_zip_code_prefix: 0
customer_city: 0
customer_state: 0


Customer_ID Unique: 100.0% (99441 unique)
Customer_Unique_ID: 96.63619633752677% (96096 unique)
Meaning:
A single 'Buy' click (Cart) is shredded into multiple customer_id's if there are multiple sellers.
This is why there are duplicate customer_unique_id's which more customer_id's

Example: 1 Cart + 3 Sellers = 3 customer_id's + 3 order_id's + 1 customer_unique_id.

--- PIPELINE STRATEGY ---
Bronze/Silver: Use customer_id/order_id to preserve the Seller-to-Item relationship (1:1:1:1).
Gold (Transactions): Group by [customer_unique_id + timestamp] to see 'True Carts'.
Gold (LTV): Group by [customer_unique_id] ONLY to see 'Lifetime Value' across all time.

Customer City & State Counts:


,count
customer_city,
sao paulo,15540
rio de janeiro,6882
belo horizonte,2773
brasilia,2131
curitiba,1521
...,...
olhos d'agua,1
pacotuba,1
sao sebastiao do paraiba,1


,count
customer_state,
SP,41746
RJ,12852
MG,11635
RS,5466
PR,5045
SC,3637
BA,3380
DF,2140
ES,2033


---
**olist_sellers_dataset**:


In [20]:
sellers = pd.read_csv("olist_sellers_dataset.csv")
sellers

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP
...,...,...,...,...
3090,98dddbc4601dd4443ca174359b237166,87111,sarandi,PR
3091,f8201cab383e484733266d1906e2fdfa,88137,palhoca,SC
3092,74871d19219c7d518d0090283e03c137,4650,sao paulo,SP
3093,e603cf3fec55f8697c9059638d6c8eb5,96080,pelotas,RS


In [21]:
print("sellers_dataset:\n")
print(sellers.columns.tolist())
print("\n")

print("Null Count:")
for col in sellers.columns.tolist():
  print(f'{col}: {sellers[col].isna().sum()}')
print("\n")

print(f'Seller_ID Unique: {sellers['seller_id'].nunique()/sellers.shape[0] * 100}%')
print("\n")

display(sellers['seller_city'].value_counts())
display(sellers['seller_state'].value_counts())

sellers_dataset:

['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state']


Null Count:
seller_id: 0
seller_zip_code_prefix: 0
seller_city: 0
seller_state: 0


Seller_ID Unique: 100.0%




,count
seller_city,
sao paulo,694
curitiba,127
rio de janeiro,96
belo horizonte,68
ribeirao preto,52
...,...
ipua,1
muqui,1
timoteo,1


,count
seller_state,
SP,1849
PR,349
MG,244
SC,190
RJ,171
RS,129
GO,40
DF,30
ES,23


---
**olist_products_dataset**:

In [22]:
products = pd.read_csv("olist_products_dataset.csv")
products

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0
...,...,...,...,...,...,...,...,...,...
32946,a0b7d5a992ccda646f2d34e418fff5a0,moveis_decoracao,45.0,67.0,2.0,12300.0,40.0,40.0,40.0
32947,bf4538d88321d0fd4412a93c974510e6,construcao_ferramentas_iluminacao,41.0,971.0,1.0,1700.0,16.0,19.0,16.0
32948,9a7c6041fa9592d9d9ef6cfe62a71f8c,cama_mesa_banho,50.0,799.0,1.0,1400.0,27.0,7.0,27.0
32949,83808703fc0706a22e264b9d75f04a2e,informatica_acessorios,60.0,156.0,2.0,700.0,31.0,13.0,20.0


In [23]:
print("products_dataset:\n")
print(products.columns.tolist())
print(products.dtypes)
print("\n")

print("Null Count:")
for col in products.columns.tolist():
  print(f'{col}: {products[col].isna().sum()}')
print("\n")

print(f'Product_ID Unique: {products['product_id'].nunique()/products.shape[0] * 100}%')
display(products[products['product_category_name'].isna()])
print("Note: 610 products have null as their category name")
print("Fix: Label the category of these products as \"Uncategorized\"")

display(products['product_category_name'].value_counts())

products_dataset:

['product_id', 'product_category_name', 'product_name_lenght', 'product_description_lenght', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']
product_id                     object
product_category_name          object
product_name_lenght           float64
product_description_lenght    float64
product_photos_qty            float64
product_weight_g              float64
product_length_cm             float64
product_height_cm             float64
product_width_cm              float64
dtype: object


Null Count:
product_id: 0
product_category_name: 610
product_name_lenght: 610
product_description_lenght: 610
product_photos_qty: 610
product_weight_g: 2
product_length_cm: 2
product_height_cm: 2
product_width_cm: 2


Product_ID Unique: 100.0%


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
105,a41e356c76fab66334f36de622ecbd3a,NaN,NaN,NaN,NaN,650.0,17.0,14.0,12.0
128,d8dee61c2034d6d075997acef1870e9b,NaN,NaN,NaN,NaN,300.0,16.0,7.0,20.0
145,56139431d72cd51f19eb9f7dae4d1617,NaN,NaN,NaN,NaN,200.0,20.0,20.0,20.0
154,46b48281eb6d663ced748f324108c733,NaN,NaN,NaN,NaN,18500.0,41.0,30.0,41.0
197,5fb61f482620cb672f5e586bb132eae9,NaN,NaN,NaN,NaN,300.0,35.0,7.0,12.0
...,...,...,...,...,...,...,...,...,...
32515,b0a0c5dd78e644373b199380612c350a,NaN,NaN,NaN,NaN,1800.0,30.0,20.0,70.0
32589,10dbe0fbaa2c505123c17fdc34a63c56,NaN,NaN,NaN,NaN,800.0,30.0,10.0,23.0
32616,bd2ada37b58ae94cc838b9c0569fecd8,NaN,NaN,NaN,NaN,200.0,21.0,8.0,16.0
32772,fa51e914046aab32764c41356b9d4ea4,NaN,NaN,NaN,NaN,1300.0,45.0,16.0,45.0


Note: 610 products have null as their category name
Fix: Label the category of these products as "Uncategorized"


,count
product_category_name,
cama_mesa_banho,3029
esporte_lazer,2867
moveis_decoracao,2657
beleza_saude,2444
utilidades_domesticas,2335
...,...
fashion_roupa_infanto_juvenil,5
casa_conforto_2,5
pc_gamer,3


---
**olist_orders_dataset**:


In [24]:
orders = pd.read_csv("olist_orders_dataset.csv")
orders

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00
...,...,...,...,...,...,...,...,...
99436,9c5dedf39a927c1b2549525ed64a053c,39bd1228ee8140590ac3aca26f2dfe00,delivered,2017-03-09 09:54:05,2017-03-09 09:54:05,2017-03-10 11:18:03,2017-03-17 15:08:01,2017-03-28 00:00:00
99437,63943bddc261676b46f01ca7ac2f7bd8,1fca14ff2861355f6e5f14306ff977a7,delivered,2018-02-06 12:58:58,2018-02-06 13:10:37,2018-02-07 23:22:42,2018-02-28 17:37:56,2018-03-02 00:00:00
99438,83c1379a015df1e13d02aae0204711ab,1aa71eb042121263aafbe80c1b562c9c,delivered,2017-08-27 14:46:43,2017-08-27 15:04:16,2017-08-28 20:52:26,2017-09-21 11:24:17,2017-09-27 00:00:00
99439,11c177c8e97725db2631073c19f07b62,b331b74b18dc79bcdf6532d51e1637c1,delivered,2018-01-08 21:28:27,2018-01-08 21:36:21,2018-01-12 15:35:03,2018-01-25 23:32:54,2018-02-15 00:00:00


In [25]:
print("orders_dataset:\n")
print(orders.columns.tolist())
display(orders.dtypes)

print("\n")

print("Null Count:")
for col in orders.columns.tolist():
  print(f'{col}: {orders[col].isna().sum()}')
print("\n")

display(orders[orders['order_id'] == '000229ec398224ef6ca0657da4fc703e'])
print(f'Order_ID Unique: {orders['order_id'].nunique()/orders.shape[0] * 100}% ({orders['order_id'].nunique()} unique)')
print(f'Customer_ID Unique: {orders['order_id'].nunique()/orders.shape[0] * 100}% ({customers['customer_id'].nunique()} unique)')
print("Note: Remember Customer_ID refers to the customer order session ID (always unique)")
print("\n")

print("Order Status Counts:")
display(orders['order_status'].value_counts())
print(f'Order Status Nulls: {orders['order_status'].isna().sum()}')
print("\n")

print("TimeStamp Column Null Counts:")
for col in ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']:
  print(f'{col}: {orders[col].isna().sum()}')

print("""
0 Nulls (purchase_timestamp): The entry point. Every order in this file has been "placed."
160 Nulls (approved_at): Orders waiting for payment clearance (Credit card processing or "Boleto" bank sync).
1,783 Nulls (carrier_date): The "Logistics Gap." These orders are paid but haven't left the seller's warehouse yet.
2,965 Nulls (customer_date): The "Last Mile." These are packages currently on a truck in Brazil.
0 Nulls (estimated_date): The moment a customer clicks "Buy," the system calculates a promise date.

Use: In your Gold layer, you will create a column: is_late = delivered_customer_date > estimated_delivery_date.
Because estimated has 0 nulls, your "Late" calculation will only return a value once the customer_date is filled in by a later CDC (Change Data Capture) update.

Since nearly 3% of your data (customer_date) is "To Be Determined," your Silver script must be able to update these rows later without creating duplicates.

How the Delta MERGE handles this:

Monday: Order 123 arrives in Bronze. delivered_customer_date is NULL. The script inserts it into Silver.
Thursday: The package is delivered. A new version of Order 123 hits Bronze with the Timestamp filled.
The MERGE: Spark sees that order_id 123 already exists in Silver. It executes whenMatchedUpdateAll(), effectively "filling in the blank" for that specific row.
""")



orders_dataset:

['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']


,0
order_id,object
customer_id,object
order_status,object
order_purchase_timestamp,object
order_approved_at,object
order_delivered_carrier_date,object
order_delivered_customer_date,object
order_estimated_delivery_date,object




Null Count:
order_id: 0
customer_id: 0
order_status: 0
order_purchase_timestamp: 0
order_approved_at: 160
order_delivered_carrier_date: 1783
order_delivered_customer_date: 2965
order_estimated_delivery_date: 0




,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
6298,000229ec398224ef6ca0657da4fc703e,6489ae5e4333f3693df5ad4372dab6d3,delivered,2018-01-14 14:33:31,2018-01-14 14:48:30,2018-01-16 12:36:48,2018-01-22 13:19:16,2018-02-05 00:00:00


Order_ID Unique: 100.0% (99441 unique)
Customer_ID Unique: 100.0% (99441 unique)
Note: Remember Customer_ID refers to the customer order session ID (always unique)


Order Status Counts:


,count
order_status,
delivered,96478
shipped,1107
canceled,625
unavailable,609
invoiced,314
processing,301
created,5
approved,2


Order Status Nulls: 0


TimeStamp Column Null Counts:
order_purchase_timestamp: 0
order_approved_at: 160
order_delivered_carrier_date: 1783
order_delivered_customer_date: 2965
order_estimated_delivery_date: 0

0 Nulls (purchase_timestamp): The entry point. Every order in this file has been "placed."
160 Nulls (approved_at): Orders waiting for payment clearance (Credit card processing or "Boleto" bank sync).
1,783 Nulls (carrier_date): The "Logistics Gap." These orders are paid but haven't left the seller's warehouse yet.
2,965 Nulls (customer_date): The "Last Mile." These are packages currently on a truck in Brazil.
0 Nulls (estimated_date): The moment a customer clicks "Buy," the system calculates a promise date.

Use: In your Gold layer, you will create a column: is_late = delivered_customer_date > estimated_delivery_date.
Because estimated has 0 nulls, your "Late" calculation will only return a value once the customer_date is filled in by a later CDC (Change Data Capture) update.

S

---
**olist_order_items_dataset**:

In [26]:
order_items = pd.read_csv("olist_order_items_dataset.csv")
order_items

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14
...,...,...,...,...,...,...,...
112645,fffc94f6ce00a00581880bf54a75a037,1,4aa6014eceb682077f9dc4bffebc05b0,b8bc237ba3788b23da09c0f1f3a3288c,2018-05-02 04:11:01,299.99,43.41
112646,fffcd46ef2263f404302a634eb57f7eb,1,32e07fd915822b0765e448c4dd74c828,f3c38ab652836d21de61fb8314b69182,2018-07-20 04:31:48,350.00,36.53
112647,fffce4705a9662cd70adb13d4a31832d,1,72a30483855e2eafc67aee5dc2560482,c3cfdc648177fdbbbb35635a37472c53,2017-10-30 17:14:25,99.90,16.95
112648,fffe18544ffabc95dfada21779c9644f,1,9c422a519119dcad7575db5af1ba540e,2b3e4a2a3ea8e01938cabda2a3e5cc79,2017-08-21 00:04:32,55.99,8.72


In [11]:
print("order_items_dataset:\n")
print(order_items.columns.tolist())
display(order_items.dtypes)
print("\n")

print("Null Count:")
for col in order_items.columns.tolist():
  print(f'{col}: {order_items[col].isna().sum()}')
print("\n")

print(f'Order_ID Unique: {order_items['order_id'].nunique()/order_items.shape[0] * 100}% ({order_items['order_id'].nunique()} unique)')
print("""
Note:

Problem: The number of unique order_id's in orders vs order_items has a delta of 775 orders that have no matching items
Reasons:
1. Abondoned/Canceled order
2. "Ghost" order (production pipeline missed the order)
3. Non-tangible orders (i.e., admin order of some kind with no actual product)

Total Orders Placed: 99,441 (From orders.csv)
Total Orders Fulfilled: 98,666 (From order_items.csv)
Cancellation/Failure Rate: The delta between the two (approx. 0.78%).

Fix: The Join Strategy (Silver Layer)
1. Left Join (`orders` -> `order_items`):
   * Use: Funnel Analysis.
   * Insight: Captures the 775 "Ghost Orders" to analyze payment/system failures.
2. Inner Join (`orders` -> `order_items`):
   * Use: Revenue & Logistics.
   * Insight: Removes noise. If there's no item, there's no sale.

Additional Note:
1 row = 1 unit (customer buys 3 cellphones = 3 rows)

A unique order_id could mean two things:
1. A customer bought a single item
2. A customer bought multiple items from different sellers
REMEBER: 1 cart can have multiple seller products (n sellers splits cart into n order_id's)

A duplicated order_id could mean two things:
1. The customer bought multiples of the same products from the same seller
2. The customer bought multiples of different products from the same seller""")
print("\n")

print("Freight_Value Analysis:")
display(order_items['freight_value'].describe())
print("Note: freight_value represents the shipping cost associated with each ITEM")
print("\n")

print("""Example:
order_id   order_item_id    product_id    seller_id  price  freight_price
ORDER_AAA              1   TOASTER_XYZ   SELLER_123  50.00          20.00
ORDER_AAA              2  COFFEE_BEANS   SELLER_123  15.00          10.00
ORDER_AAA              3      MUG_BLUE   SELLER_123  10.00           5.00
""")


order_items_dataset:

['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']


,0
order_id,object
order_item_id,int64
product_id,object
seller_id,object
shipping_limit_date,object
price,float64
freight_value,float64




Null Count:
order_id: 0
order_item_id: 0
product_id: 0
seller_id: 0
shipping_limit_date: 0
price: 0
freight_value: 0


Order_ID Unique: 87.58632933865957% (98666 unique)

Note:

Problem: The number of unique order_id's in orders vs order_items has a delta of 775 orders that have no matching items
Reasons:
1. Abondoned/Canceled order
2. "Ghost" order (production pipeline missed the order)
3. Non-tangible orders (i.e., admin order of some kind with no actual product)

Total Orders Placed: 99,441 (From orders.csv)
Total Orders Fulfilled: 98,666 (From order_items.csv)
Cancellation/Failure Rate: The delta between the two (approx. 0.78%).

Fix: The Join Strategy (Silver Layer)
1. Left Join (`orders` -> `order_items`):
   * Use: Funnel Analysis.
   * Insight: Captures the 775 "Ghost Orders" to analyze payment/system failures.
2. Inner Join (`orders` -> `order_items`):
   * Use: Revenue & Logistics.
   * Insight: Removes noise. If there's no item, there's no sale.

Additional Note:
1 row = 1

,freight_value
count,112650.000000
mean,19.990320
std,15.806405
min,0.000000
25%,13.080000
50%,16.260000
75%,21.150000
max,409.680000


Note: freight_value represents the shipping cost associated with each ITEM


Example:
order_id   order_item_id    product_id    seller_id  price  freight_price
ORDER_AAA              1   TOASTER_XYZ   SELLER_123  50.00          20.00
ORDER_AAA              2  COFFEE_BEANS   SELLER_123  15.00          10.00
ORDER_AAA              3      MUG_BLUE   SELLER_123  10.00           5.00



---
**olist_order_payments_dataset**:

In [12]:
order_payments = pd.read_csv("olist_order_payments_dataset.csv")
order_payments

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45
...,...,...,...,...,...
103881,0406037ad97740d563a178ecc7a2075c,1,boleto,1,363.31
103882,7b905861d7c825891d6347454ea7863f,1,credit_card,2,96.80
103883,32609bbb3dd69b3c066a6860554a77bf,1,credit_card,1,47.77
103884,b8b61059626efa996a60be9bb9320e10,1,credit_card,5,369.54


In [13]:
print("order_payments_dataset:\n")
print(order_payments.columns.tolist())
display(order_payments.dtypes)
print("\n")

print("Null Count:")
for col in order_payments.columns.tolist():
  print(f'{col}: {order_payments[col].isna().sum()}')
print("\n")

print(f'Order_ID Unique: {order_payments['order_id'].nunique()/order_payments.shape[0] * 100}% ({order_payments['order_id'].nunique()} unique)')
print("""
Note:
4.3% of your orders involved more than one payment entry.
Row 1: order_id_abc, payment_sequential = 1, type = voucher    , value = 10.00
Row 2: order_id_abc, payment_sequential = 2, type = credit_card, value = 90.00

Additional Note:
Problem: There are 99,441 unique order_id's in the orders table, but only 99,440 unique order_id's in the payments table. This 1 order_id is a "Free Order" or a "Data Orphan."
Reasons:
1. 100% Dicsount/Voucher
2. System Error
3. Test order
Fix: Use a LEFT JOIN to ensure this order is not lost
""")

display(order_payments['payment_type'].value_counts())
display(order_payments['payment_value'].describe())

order_payments_dataset:

['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']


,0
order_id,object
payment_sequential,int64
payment_type,object
payment_installments,int64
payment_value,float64




Null Count:
order_id: 0
payment_sequential: 0
payment_type: 0
payment_installments: 0
payment_value: 0


Order_ID Unique: 95.72030880003081% (99440 unique)

Note:
4.3% of your orders involved more than one payment entry.
Row 1: order_id_abc, payment_sequential = 1, type = voucher    , value = 10.00
Row 2: order_id_abc, payment_sequential = 2, type = credit_card, value = 90.00

Additional Note:
Problem: There are 99,441 unique order_id's in the orders table, but only 99,440 unique order_id's in the payments table. This 1 order_id is a "Free Order" or a "Data Orphan."
Reasons:
1. 100% Dicsount/Voucher
2. System Error
3. Test order
Fix: Use a LEFT JOIN to ensure this order is not lost



,count
payment_type,
credit_card,76795
boleto,19784
voucher,5775
debit_card,1529
not_defined,3


,payment_value
count,103886.000000
mean,154.100380
std,217.494064
min,0.000000
25%,56.790000
50%,100.000000
75%,171.837500
max,13664.080000


---
**olist_order_reviews_dataset**:

In [14]:
order_reviews = pd.read_csv("olist_order_reviews_dataset.csv")
order_reviews

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01 00:00:00,2018-03-02 10:26:53
...,...,...,...,...,...,...,...
99219,574ed12dd733e5fa530cfd4bbf39d7c9,2a8c23fee101d4d5662fa670396eb8da,5,NaN,NaN,2018-07-07 00:00:00,2018-07-14 17:18:30
99220,f3897127253a9592a73be9bdfdf4ed7a,22ec9f0669f784db00fa86d035cf8602,5,NaN,NaN,2017-12-09 00:00:00,2017-12-11 20:06:42
99221,b3de70c89b1510c4cd3d0649fd302472,55d4004744368f5571d1f590031933e4,5,NaN,"Excelente mochila, entrega super rápida. Super...",2018-03-22 00:00:00,2018-03-23 09:10:43
99222,1adeb9d84d72fe4e337617733eb85149,7725825d039fc1f0ceb7635e3f7d9206,4,NaN,NaN,2018-07-01 00:00:00,2018-07-02 12:59:13


In [15]:
print("order_reviews_dataset:\n")
print(order_reviews.columns.tolist())
print("\n")

print("Null Count:")
for col in order_reviews.columns.tolist():
  print(f'{col}: {order_reviews[col].isna().sum()}')
print("\n")

print(f'Review_ID Unique: {order_reviews['review_id'].nunique()/order_reviews.shape[0] * 100}% ({order_reviews['review_id'].nunique()} unique)')
print(f'Order_ID Unique: {order_reviews['order_id'].nunique()/order_reviews.shape[0] * 100}% ({order_reviews['order_id'].nunique()} unique)')
print("""
Note:
Not all order_id's got reviews

Duplicate review_id's:
  * 1 review_id can span n order_id's:
  * Caused by a customer leaving a single review for the "box"
    which can have items from n sellers.

Duplicate order_id's:
  * 1 order_id can span at least 1 review_id:
  * Caused by Re-submissions or Customer Support interactions.

Fix: Treat the review_id as the primary key for "Customer Happiness.""")
print("\n")

print("Review_Score Analysis:")
display(order_reviews['review_score'].describe())

order_reviews_dataset:

['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']


Null Count:
review_id: 0
order_id: 0
review_score: 0
review_comment_title: 87656
review_comment_message: 58247
review_creation_date: 0
review_answer_timestamp: 0


Review_ID Unique: 99.17963395952593% (98410 unique)
Order_ID Unique: 99.44469080061276% (98673 unique)

Note:
Not all order_id's got reviews

Duplicate review_id's:
  * 1 review_id can span n order_id's:
  * Caused by a customer leaving a single review for the "box"
    which can have items from n sellers.

Duplicate order_id's:
  * 1 order_id can span at least 1 review_id:
  * Caused by Re-submissions or Customer Support interactions.

Fix: Treat the review_id as the primary key for "Customer Happiness.


Review_Score Analysis:


,review_score
count,99224.000000
mean,4.086421
std,1.347579
min,1.000000
25%,4.000000
50%,5.000000
75%,5.000000
max,5.000000


---
**olist_category_name_translation_dataset**:

In [16]:
print("category_name_translation:\n")
category_name_translation = pd.read_csv("product_category_name_translation.csv")
category_name_translation

category_name_translation:



,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor
...,...,...
66,flores,flowers
67,artes_e_artesanato,arts_and_craftmanship
68,fraldas_higiene,diapers_and_hygiene
69,fashion_roupa_infanto_juvenil,fashion_childrens_clothes
